# 🛩️ Example: Working with the LAPAN Surveillance Aircraft Model (LSU-05 NG)

This notebook demonstrates the usage of the linear longitudinal model of the LAPAN LSU-05 NG surveillance aircraft in the TensorAeroSpace environment.

## 📋 What we will do:
1. Import the required libraries
2. Configure time parameters and the reference signal
3. Create and initialize the environment
4. Execute one simulation step
5. Analyze the results

## 📚 Importing Libraries

Loading all necessary modules for working with the aircraft model:

In [1]:
# Core libraries
import gymnasium as gym
import numpy as np

# Import TensorAeroSpace to register environments
import tensoraerospace

# TensorAeroSpace modules
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

## ⚙️ Simulation Parameters Setup

Defining time parameters and creating a reference signal for pitch angle control:

In [ ]:
# Discretization parameters
dt = 0.01  # Discretization step (seconds)

# Generate time period
tp = generate_time_period(tn=20, dt=dt)  # 20 seconds of simulation
tps = convert_tp_to_sec_tp(tp, dt=dt)    # Convert to seconds
number_time_steps = len(tp)             # Total number of time steps

# Create a step reference signal for the pitch angle (theta)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10*dt, output_rad=True), 
    [1, -1]
)

print(f"📊 Simulation parameters:")
print(f"   • Simulation time: {tp[-1]:.1f} sec")
print(f"   • Discretization step: {dt} sec")
print(f"   • Number of steps: {number_time_steps}")
print(f"   • Reference signal shape: {reference_signals.shape}")

## 🚀 Creating and Initializing the Environment

Creating the LAPAN LSU-05 NG environment with the specified parameters and performing a reset:

In [3]:
# Create the LAPAN LSU-05 NG environment
env = gym.make(
    "LinearLongitudinalLAPAN-v0",
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0], [0]],  # Initial state [u, w, q, theta]
    reference_signal=reference_signals,
    tracking_states=["theta"]
)

# Initialize the environment
state, info = env.reset()

print(f"✅ Environment successfully created and initialized!")
print(f"📈 Initial state: {state.flatten()}")
print(f"🎯 State space shape: {env.observation_space.shape}")
print(f"🎮 Action space shape: {env.action_space.shape}")
print(f"📊 Tracking states: {env.unwrapped.tracking_states}")
print(f"📋 State space: {env.unwrapped.state_space}")
print(f"📤 Output space: {env.unwrapped.output_space}")

✅ Environment successfully created and initialized!
📈 Initial state: [0. 0. 0. 0.]
🎯 State space shape: (2, 1)
🎮 Action space shape: (1, 1)
📊 Tracking states: ['theta']
📋 State space: ['theta', 'q']
📤 Output space: ['theta', 'q']


/home/mr8bit/.pyenv/versions/3.12.7/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


## 🎮 Executing a Simulation Step

Applying a control action and observing the system response:

In [ ]:
# Apply a control action
control_input = np.array([1.0], dtype=np.float32)

# Execute one simulation step
state, reward, terminated, truncated, info = env.step(control_input)

print(f"🎯 Control action: {control_input[0]:.2f} degrees")
print(f"📊 New state: {state.flatten()}")
theta_idx = env.unwrapped.state_space.index("theta")
q_idx = env.unwrapped.state_space.index("q")
print(f"   • Pitch angle (theta): {np.rad2deg(state[theta_idx]):.4f} deg")
print(f"   • Angular velocity (q): {np.rad2deg(state[q_idx]):.4f} deg/s")
# Process reward (may be an array or scalar)
reward_value = reward
print(f"🏆 Reward: {reward_value:.6f}")
print(f"🔚 Terminated: {terminated}")
print(f"⏰ Truncated: {truncated}")

## Visualization: Full Simulation with P-Controller

In [ ]:
import matplotlib.pyplot as plt

# Create fresh environment for visualization
env_vis = gym.make(
    "LinearLongitudinalLAPAN-v0",
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0], [0]],
    reference_signal=reference_signals,
    tracking_states=["theta"]
)
obs, _ = env_vis.reset()

# Run simulation with proportional control
# LAPAN env: action passed directly to model (radians internally)
# obs shape (2,): [theta, q] in radians from model
rewards_hist = []
for step in range(number_time_steps - 2):
    ref_val = reference_signals[0, min(step, reference_signals.shape[1]-1)]  # radians
    state_val = obs[0]  # theta in radians
    error = ref_val - state_val
    control = np.clip(error * 0.5, -0.5, 0.5)  # radians
    obs, reward, terminated, truncated, _ = env_vis.step(np.array([control], dtype=np.float32))
    rewards_hist.append(reward)
    if terminated or truncated:
        break

# Get history from model
model = env_vis.unwrapped.model
theta_hist = model.get_state('theta', to_deg=True)
q_hist = model.get_state('q', to_deg=True)
control_hist = model.get_control('ele', to_deg=True)

# Reference signal in degrees
n = len(theta_hist)
ref_deg = np.rad2deg(reference_signals[0, :n])
time = tps[:n]

# Create figure
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('LAPAN LSU-05 NG -- Step Response with P-Controller', fontsize=14, fontweight='bold')

# Plot 1: Tracking response
axes[0,0].plot(time, ref_deg, 'r--', linewidth=2, label='Reference')
axes[0,0].plot(time, theta_hist, 'b-', linewidth=1.5, label='Response')
ref_nonzero = ref_deg != 0
if np.any(ref_nonzero):
    axes[0,0].fill_between(time, ref_deg*0.95, ref_deg*1.05, alpha=0.1, color='green', label='$\\pm$5% band')
axes[0,0].set_xlabel('Time (s)')
axes[0,0].set_ylabel('Angle (deg)')
axes[0,0].set_title('Pitch Angle Tracking ($\\theta$)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Control input
axes[0,1].plot(time, control_hist, 'r-', linewidth=1.5)
axes[0,1].set_xlabel('Time (s)')
axes[0,1].set_ylabel('Deflection (deg)')
axes[0,1].set_title('Elevator Deflection')
axes[0,1].grid(True, alpha=0.3)
axes[0,1].axhline(y=0, color='k', linestyle='--', alpha=0.3)

# Plot 3: Tracking error
error_hist = ref_deg - theta_hist
axes[1,0].plot(time, error_hist, 'g-', linewidth=1.5)
axes[1,0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1,0].set_xlabel('Time (s)')
axes[1,0].set_ylabel('Error (deg)')
axes[1,0].set_title('Tracking Error')
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Reward
time_r = tps[:len(rewards_hist)]
axes[1,1].plot(time_r, rewards_hist, 'k-', linewidth=1, alpha=0.5, label='Instant')
win = min(50, max(1, len(rewards_hist)//20))
if len(rewards_hist) > win:
    smoothed = np.convolve(rewards_hist, np.ones(win)/win, mode='same')
    axes[1,1].plot(time_r, smoothed, 'b-', linewidth=2, label=f'Moving avg (w={win})')
axes[1,1].set_xlabel('Time (s)')
axes[1,1].set_ylabel('Reward')
axes[1,1].set_title(f'Reward (total: {sum(rewards_hist):.1f})')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Additional: Pitch rate plot
fig2, ax2 = plt.subplots(figsize=(10, 3))
ax2.plot(time, q_hist, 'purple', linewidth=1.5)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Pitch Rate (deg/s)')
ax2.set_title('LAPAN LSU-05 NG -- Pitch Rate ($q$)')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()